In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../'))

import numpy as np
import tensorflow as tf
import random

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

from src.rnn.layers import Embedding, SimpleRNNCell, LSTMCell, DenseProjection, DenseOutput
print('Import OK')

## 1. Embedding Layer

In [ ]:
VOCAB_SIZE = 100
EMBED_DIM  = 32

keras_emb = tf.keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM)

token_ids = np.array([3, 7, 12, 0, 45])
_ = keras_emb(token_ids)  # build weights

scratch_emb = Embedding.from_keras(keras_emb)

keras_out  = keras_emb(token_ids).numpy()
scratch_out = scratch_emb.forward(token_ids)

print(f'Keras   shape: {keras_out.shape}')
print(f'Scratch shape: {scratch_out.shape}')
print(f'Max diff: {np.max(np.abs(keras_out - scratch_out)):.2e}')
assert np.allclose(keras_out, scratch_out, atol=1e-6)
print('PASS: Embedding')

## 2. SimpleRNNCell

In [ ]:
HIDDEN = 64
T = 5
N = 2

keras_rnn = tf.keras.layers.SimpleRNN(HIDDEN, return_sequences=True, return_state=True)

x_rnn = np.random.randn(N, T, EMBED_DIM).astype(np.float32)
keras_seq, keras_h = keras_rnn(x_rnn)
keras_seq, keras_h = keras_seq.numpy(), keras_h.numpy()

scratch_rnn = SimpleRNNCell.from_keras(keras_rnn)
scratch_seq, scratch_h = scratch_rnn.forward(x_rnn, return_sequences=True)

print(f'Seq  max diff: {np.max(np.abs(keras_seq - scratch_seq)):.2e}')
print(f'h    max diff: {np.max(np.abs(keras_h  - scratch_h )):.2e}')
assert np.allclose(keras_seq, scratch_seq, atol=1e-5)
assert np.allclose(keras_h,   scratch_h,   atol=1e-5)
print('PASS: SimpleRNNCell (sequences + final state)')

In [ ]:
# Test step() yang dipakai greedy decoding
h = np.zeros((N, HIDDEN), dtype=np.float32)
for t in range(T):
    h_keras  = keras_rnn.cell(x_rnn[:, t, :], [h])[0].numpy()
    h_scratch = scratch_rnn.step(x_rnn[:, t, :], h)
    assert np.allclose(h_keras, h_scratch, atol=1e-5), f'step mismatch at t={t}'
    h = h_scratch
print('PASS: SimpleRNNCell.step() (greedy decoding compatible)')

## 3. LSTMCell

In [ ]:
keras_lstm = tf.keras.layers.LSTM(HIDDEN, return_sequences=True, return_state=True)

x_lstm = np.random.randn(N, T, EMBED_DIM).astype(np.float32)
keras_seq_l, keras_h_l, keras_c_l = keras_lstm(x_lstm)
keras_seq_l = keras_seq_l.numpy()
keras_h_l, keras_c_l = keras_h_l.numpy(), keras_c_l.numpy()

scratch_lstm = LSTMCell.from_keras(keras_lstm)
scratch_seq_l, scratch_h_l, scratch_c_l = scratch_lstm.forward(x_lstm, return_sequences=True)

print(f'Seq  max diff: {np.max(np.abs(keras_seq_l - scratch_seq_l)):.2e}')
print(f'h    max diff: {np.max(np.abs(keras_h_l  - scratch_h_l )):.2e}')
print(f'c    max diff: {np.max(np.abs(keras_c_l  - scratch_c_l )):.2e}')
assert np.allclose(keras_seq_l, scratch_seq_l, atol=1e-5)
assert np.allclose(keras_h_l,   scratch_h_l,   atol=1e-5)
assert np.allclose(keras_c_l,   scratch_c_l,   atol=1e-5)
print('PASS: LSTMCell (sequences + h + c states)')

In [ ]:
# Test step() yang dipakai greedy decoding
h = np.zeros((N, HIDDEN), dtype=np.float32)
c = np.zeros((N, HIDDEN), dtype=np.float32)
for t in range(T):
    h_k, c_k = keras_lstm.cell(x_lstm[:, t, :], [h, c])[:2]
    h_k, c_k = h_k.numpy(), c_k.numpy()
    h_s, c_s = scratch_lstm.step(x_lstm[:, t, :], h, c)
    assert np.allclose(h_k, h_s, atol=1e-5), f'h mismatch t={t}'
    assert np.allclose(c_k, c_s, atol=1e-5), f'c mismatch t={t}'
    h, c = h_s, c_s
print('PASS: LSTMCell.step() (greedy decoding compatible)')

## 4. DenseProjection & DenseOutput

In [ ]:
FEATURE_DIM = 2048

# DenseProjection: feature → embed_dim (no activation by default)
keras_proj = tf.keras.layers.Dense(EMBED_DIM, name='image_projection')
x_feat = np.random.randn(N, FEATURE_DIM).astype(np.float32)
keras_proj_out = keras_proj(x_feat).numpy()

scratch_proj = DenseProjection.from_keras(keras_proj)
scratch_proj_out = scratch_proj.forward(x_feat)

print(f'DenseProjection diff: {np.max(np.abs(keras_proj_out - scratch_proj_out)):.2e}')
assert np.allclose(keras_proj_out, scratch_proj_out, atol=1e-5)
print('PASS: DenseProjection')

# DenseOutput: hidden → vocab_size (softmax)
VOCAB_SIZE_OUT = 200
keras_out_layer = tf.keras.layers.Dense(VOCAB_SIZE_OUT, activation='softmax', name='output_dense')
x_hidden = np.random.randn(N, HIDDEN).astype(np.float32)
keras_out_out = keras_out_layer(x_hidden).numpy()

scratch_out_layer = DenseOutput.from_keras(keras_out_layer)
scratch_out_out = scratch_out_layer.forward(x_hidden)

print(f'DenseOutput diff: {np.max(np.abs(keras_out_out - scratch_out_out)):.2e}')
assert np.allclose(keras_out_out, scratch_out_out, atol=1e-5)
print('PASS: DenseOutput (softmax)')

## 5. End-to-End: Pre-inject Greedy Decode Step

In [ ]:
from src.rnn.models import build_rnn_decoder_pre_inject
from src.utils.weight_loader import load_scratch_components
from src.pipeline.greedy_decode import greedy_decode_rnn, greedy_decode_lstm

VOCAB_SIZE_E2E = 50
EMBED_E2E = 16
HIDDEN_E2E = 32
FEAT_E2E = 64

vocab = {'<pad>': 0, '<start>': 1, '<end>': 2, '<unk>': 3}
for i in range(4, VOCAB_SIZE_E2E):
    vocab[f'word{i}'] = i
idx2word = {v: k for k, v in vocab.items()}

for rnn_type in ['rnn', 'lstm']:
    model = build_rnn_decoder_pre_inject(
        vocab_size=VOCAB_SIZE_E2E,
        embed_dim=EMBED_E2E,
        rnn_units=HIDDEN_E2E,
        feature_dim=FEAT_E2E,
        num_rnn_layers=2,
        rnn_type=rnn_type,
    )

    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, f'{rnn_type}_test.h5')
        model.save(path)
        components = load_scratch_components(path, rnn_type)

    feature = np.random.randn(FEAT_E2E).astype(np.float32)
    decode_fn = greedy_decode_lstm if rnn_type == 'lstm' else greedy_decode_rnn
    caption = decode_fn(
        feature,
        components['cnn_projection'],
        components['embedding'],
        components['rnn_cells'],
        components['output_dense'],
        vocab, idx2word,
        max_len=10,
    )
    print(f'PASS: greedy_decode_{rnn_type}() → "{caption}"')

print('\nSemua unit test PASSED')